In [1]:
import pandas as pd
import numpy as np

# 1. Load Original Untouched Data
input_path = r'D:\Uni\Year_3_sem_2\XAI\Project\phase_2\Dataset.csv'
df = pd.read_csv(input_path)


In [4]:
def apply_research_paper_refinements(df):
    print("Starting Research-Backed Refinements...")
    
    # STEP 1: PHYSIOLOGICAL CLIPPING (Cleaning)
    # Removes impossible sensor noise without losing real medical 'dips' or 'spikes'
    df = df[df['Temp'].between(25, 45) | df['Temp'].isna()]
    df = df[df['SBP'].between(40, 300) | df['SBP'].isna()]
    df = df[df['HR'].between(20, 280) | df['HR'].isna()]
    df = df[df['Resp'].between(4, 80) | df['Resp'].isna()]
    
    # STEP 2: RESPONSIBLE AI FIREWALL (Leakage Prevention)
    # Remove variables that reveal doctor actions to prevent 'cheating'
    treatment_cols = ['Vasopressors', 'Antibiotics', 'Vento_Settings', 'InvasiveVent']
    df = df.drop(columns=[c for c in treatment_cols if c in df.columns])
    
    # STEP 3: EARLY MORTALITY FILTER
    # Exclude patients with < 24h of data to focus on early prediction
    patient_lengths = df.groupby('Patient_ID')['ICULOS'].max()
    valid_pids = patient_lengths[patient_lengths >= 24].index
    df = df[df['Patient_ID'].isin(valid_pids)]
    
    # STEP 4: DELTA FEATURE ENGINEERING
    # Capture the "Rate of Change" (dips and rises) for the Transformer[cite: 2]
    df = df.sort_values(['Patient_ID', 'ICULOS'])
    for feat in ['HR', 'SBP', 'Temp', 'Resp']:
        if feat in df.columns:
            df[f'{feat}_Delta'] = df.groupby('Patient_ID')[feat].diff().fillna(0)
            
    # STEP 5: OXYGEN & RISK PROXIES
    # Paper identified oxygen support and comorbidities as top predictors[cite: 2]
    if 'FiO2' in df.columns:
        df['high_oxygen_support'] = (df['FiO2'] > 0.5).astype(int)
    
    # Unit types often correlate with baseline risk[cite: 2]
    if 'Unit1' in df.columns and 'Unit2' in df.columns:
        df['baseline_risk_index'] = df['Unit1'] + df['Unit2']

    return df


In [5]:
# Execute and Save
df_engineered = apply_research_paper_refinements(df)
output_path = r'D:\Uni\Year_3_sem_2\XAI\Project\phase_2\Transformer_Engineered_Dataset.csv'
df_engineered.to_csv(output_path, index=False)

print(f"Success! Cleaned and Engineered data saved to: {output_path}")

Starting Research-Backed Refinements...
Success! Cleaned and Engineered data saved to: D:\Uni\Year_3_sem_2\XAI\Project\phase_2\Transformer_Engineered_Dataset.csv
